# DATA005 — 전체 공개 데이터셋 Google Drive 보관

MixFake, DFADD, ASVspoof5, FMA small, WaveFake v1.2,
SONICS, FakeMusicCaps v2, SpeechFake를 원본 공개 서버에서 Google Drive로 직접 전송한다.
PC와 Colab 로컬 디스크에는 대용량 원본을 통째로 저장하지 않는다.

이미 Drive에 정상 업로드된 파일은 자동으로 건너뛴다. 45 GB 안전 구간을
내부에서 반복하므로 한 번 Run All로 전체를 진행하며, 런타임이 종료되면
같은 노트북을 다시 Run All하여 이어받을 수 있다.

SONICS와 FakeMusicCaps는 `03_license_hold`에만 보관한다. 두 데이터셋은
DACON 운영진의 비상업 라이선스 사용 가능 답변 전까지 학습에 사용하지 않는다.


## 1. 전송 범위와 안전 설정

기존 Drive 파일은 이름과 크기가 모두 같은 경우에만 건너뛰며 덮어쓰지 않는다.
한 배치당 최대 45 GB를 전송하고, 중단 시 Drive 상태 파일의 resumable session으로
이어받는다. 최대 64개 배치를 자동 반복한다.


In [ ]:
DRY_RUN = False
TRANSFER_CONFIRMATION = "ARCHIVE_PUBLIC_AND_LICENSE_HOLD_TO_DRIVE"
REQUIRED_CONFIRMATION = "ARCHIVE_PUBLIC_AND_LICENSE_HOLD_TO_DRIVE"

PER_RUN_BUDGET_GB = 45
UPLOAD_CHUNK_MIB = 16
MAX_AUTOMATIC_BATCHES = 64
SELECTED_DATASETS = (
    "MixFake",
    "DFADD",
    "ASVspoof5",
    "FMA_small",
    "WaveFake_v1.2",
    "FakeMusicCaps_v2",
    "SONICS",
    "SpeechFake",
)
TARGET_ROOT_FOLDER_ID = "1vsbNmEgxwgm-k81c23wCLRpFymVk7cVt"
EXPECTED_UPLOAD_EMAIL = "jisoo5849@gmail.com"

assert TRANSFER_CONFIRMATION == REQUIRED_CONFIRMATION
print(f"Mode: {'DRY RUN' if DRY_RUN else 'TRANSFER'}")
print(f"Per-run budget: {PER_RUN_BUDGET_GB} GB")
print(f"Automatic batch limit: {MAX_AUTOMATIC_BATCHES}")
print("Datasets:", ", ".join(SELECTED_DATASETS))
print("Expected uploader:", EXPECTED_UPLOAD_EMAIL)
print("✅ 실제 보관 전송 승인 문구 확인")


Mode: TRANSFER
Per-run budget: 45 GB
Automatic batch limit: 64
Datasets: MixFake, DFADD, ASVspoof5, FMA_small, WaveFake_v1.2, FakeMusicCaps_v2, SONICS, SpeechFake
Expected uploader: jisoo5849@gmail.com
✅ 실제 보관 전송 승인 문구 확인


## 2. 전송 라이브러리 준비

Colab 기본 환경을 우선 사용하고, HTTP·Google Drive API 패키지가 빠진 경우에만
설치한다. 데이터 보관 작업에는 GPU가 필요하지 않다.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "requests": "requests",
    "google.auth": "google-auth",
    "googleapiclient": "google-api-python-client",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("✅ 전송 라이브러리 준비 완료")


✅ 전송 라이브러리 준비 완료


## 3. 본인 Google Drive 계정 인증

인증 창에서 `EXPECTED_UPLOAD_EMAIL`로 지정한 본인 계정만 선택한다.
다른 계정이 선택되면 전송 전에 중단한다. 여유 공간은 이 계정의
저장용량을 기준으로 한다.

Colab의 기본 임시 인증에서 `credential propagation was unsuccessful`이
발생하는 경우를 피하기 위해 Google의 대화형 `gcloud` 인증 경로를 사용한다.
출력된 링크를 열어 `jisoo5849@gmail.com`으로 허용하고, 요청되면
인증 코드를 셀 입력창에 넣는다.


In [ ]:
import os

# Colab front-end의 임시 credential propagation 경로를 사용하지 않고,
# 브라우저 링크/코드 방식의 gcloud 인증을 강제한다.
os.environ["USE_AUTH_EPHEM"] = "0"

from google.colab import auth
import google.auth
from googleapiclient.discovery import build

print("Google 인증을 시작합니다. 출력된 링크/코드 안내를 따라주세요.")
auth.authenticate_user(clear_output=False)
credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive"]
)
drive_service = build("drive", "v3", credentials=credentials, cache_discovery=False)
about = drive_service.about().get(fields="user,storageQuota").execute()
active_email = about["user"]["emailAddress"]
quota = about.get("storageQuota", {})
free_bytes = None
if quota.get("limit") is not None and quota.get("usage") is not None:
    free_bytes = int(quota["limit"]) - int(quota["usage"])

print("Authenticated account:", active_email)
print("Free storage (GB):", None if free_bytes is None else round(free_bytes / 1e9, 2))
assert active_email.lower() == EXPECTED_UPLOAD_EMAIL.lower(), (
    f"잘못된 계정입니다: {active_email}. "
    f"Colab 인증을 초기화하고 {EXPECTED_UPLOAD_EMAIL}로 다시 인증하세요."
)
print("✅ 본인 계정 확인 완료")


Google 인증을 시작합니다. 출력된 링크/코드 안내를 따라주세요.


KeyboardInterrupt: Interrupted by user

## 4. 고정 소스와 Drive 폴더 맵 준비

검증된 전송 모듈과 폴더 ID, 라이선스 분류 매니페스트를 런타임에 복원한다.
Hugging Face revision과 Zenodo record를 고정하여 재실행 시 원본이 바뀌지 않게 한다.


In [ ]:
from pathlib import Path
import hashlib

RUNTIME_DIR = Path("/content/deepvoice_dataset_archive")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT_PATH = RUNTIME_DIR / "drive_dataset_streamer.py"
FOLDER_MAP_PATH = RUNTIME_DIR / "drive_dataset_folder_map_20260901.csv"
CATALOG_PATH = RUNTIME_DIR / "dataset_archive_manifest_20260901.csv"

SCRIPT_PATH.write_text('"""Stream approved Deepvoice datasets from public sources into Google Drive.\n\nThe module is designed for Google Colab.  Large source files are never written to\nthe user\'s PC and are not staged wholesale in ``/content``.  A bounded in-memory\nchunk is read from the public source and forwarded to a Google Drive resumable\nupload session.\n\nSafety defaults:\n\n* dry-run is enabled unless the caller explicitly disables it;\n* at most 40 GB (decimal bytes) is transferred in one run;\n* existing destination files are skipped only when both name and size match;\n* a name collision with a different size is a hard error (nothing is overwritten);\n* regular My Drive free space is checked with a 5 GiB safety reserve;\n* a small Drive-side JSON state file records source revision, URL, licence,\n  declared checksum, progress, and the resumable-session URL.\n\nThe original P0/P1 sources plus the explicitly requested archive-only music\ndatasets are implemented here.  ``license_hold`` changes where a dataset is\nstored and whether it may be used for training; it does not prevent archival\nwhen the caller explicitly enables ``allow_license_hold_archive``.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport re\nimport time\nfrom dataclasses import asdict, dataclass, field\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable, Mapping, Sequence\nfrom urllib.parse import quote\n\nimport requests\nfrom urllib3.exceptions import HTTPError as Urllib3HTTPError\n\n\nGB = 1000**3\nGIB = 1024**3\nMIB = 1024**2\nDRIVE_FOLDER_MIME = "application/vnd.google-apps.folder"\nSTATE_MIME = "application/json"\nDEFAULT_ROOT_FOLDER_ID = "1vsbNmEgxwgm-k81c23wCLRpFymVk7cVt"\nDEFAULT_DATASETS = ("MixFake", "DFADD", "ASVspoof5", "FMA_small")\nSUPPORTED_DATASETS = DEFAULT_DATASETS + (\n    "SpeechFake",\n    "WaveFake_v1.2",\n    "SONICS",\n    "FakeMusicCaps_v2",\n)\n\n# Revisions are intentionally pinned.  Re-running the notebook therefore cannot\n# silently ingest a newer upstream dataset revision.\nHF_REVISIONS = {\n    "MixFake": "fbaf2e953daf3779559ae2ba300a53300a9616f2",\n    "DFADD": "dfc1eeab3cb0068db8e87a2b89a1ebd103665b1f",\n    "SONICS": "3788dca9f9f11ad92e9097ef4b58eee247661e7f",\n}\nMODELSCOPE_REVISIONS = {\n    "SpeechFake": "e315a544246df765336bab9595ed552520444c4a",\n}\n\nDATASET_ALIASES = {\n    "asvspoof 5": "ASVspoof5",\n    "asvspoof5": "ASVspoof5",\n    "fma small": "FMA_small",\n    "fma_small": "FMA_small",\n    "mixfake": "MixFake",\n    "dfadd": "DFADD",\n    "speechfake": "SpeechFake",\n    "wavefake": "WaveFake_v1.2",\n    "wavefake v1.2": "WaveFake_v1.2",\n    "wavefake_v1.2": "WaveFake_v1.2",\n    "sonics": "SONICS",\n    "fakemusiccaps": "FakeMusicCaps_v2",\n    "fakemusiccaps v2": "FakeMusicCaps_v2",\n    "fakemusiccaps_v2": "FakeMusicCaps_v2",\n}\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat(timespec="seconds")\n\n\ndef canonical_dataset_name(value: str) -> str:\n    key = value.strip().lower()\n    return DATASET_ALIASES.get(key, value.strip())\n\n\ndef human_bytes(value: int | None) -> str:\n    if value is None:\n        return "unknown"\n    if value == 0:\n        return "0 B"\n    units = ("B", "KiB", "MiB", "GiB", "TiB")\n    order = min(int(math.log(value, 1024)), len(units) - 1)\n    return f"{value / (1024**order):,.2f} {units[order]}"\n\n\ndef request_with_retry(\n    method: str,\n    url: str,\n    *,\n    attempts: int = 5,\n    timeout: tuple[int, int] = (30, 180),\n    session: requests.Session | None = None,\n    **kwargs: Any,\n) -> requests.Response:\n    requester = session or requests\n    last_error: Exception | None = None\n    for attempt in range(attempts):\n        try:\n            response = requester.request(method, url, timeout=timeout, **kwargs)\n            if response.status_code in {429, 500, 502, 503, 504}:\n                response.close()\n                raise requests.HTTPError(f"retryable HTTP {response.status_code}")\n            return response\n        except (requests.RequestException, OSError) as exc:\n            last_error = exc\n            if attempt + 1 >= attempts:\n                break\n            time.sleep(min(2**attempt, 20))\n    raise RuntimeError(f"request failed after {attempts} attempts: {url}") from last_error\n\n\n@dataclass(frozen=True)\nclass SourceFile:\n    dataset: str\n    relative_path: str\n    url: str\n    size: int\n    source_revision: str\n    license_name: str\n    license_url: str\n    checksum_algorithm: str | None = None\n    checksum_value: str | None = None\n    source_page: str | None = None\n\n    @property\n    def name(self) -> str:\n        return Path(self.relative_path).name\n\n    @property\n    def key(self) -> str:\n        return f"{self.dataset}::{self.source_revision}::{self.relative_path}"\n\n    @property\n    def checksum(self) -> str | None:\n        if self.checksum_algorithm and self.checksum_value:\n            return f"{self.checksum_algorithm}:{self.checksum_value}"\n        return None\n\n    def manifest_record(self) -> dict[str, Any]:\n        row = asdict(self)\n        row.update({"name": self.name, "key": self.key, "checksum": self.checksum})\n        return row\n\n\n@dataclass\nclass TransferConfig:\n    dry_run: bool = True\n    max_run_bytes: int = 40 * GB\n    chunk_bytes: int = 16 * MIB\n    state_flush_bytes: int = 256 * MIB\n    free_space_reserve_bytes: int = 5 * GIB\n    target_root_folder_id: str = DEFAULT_ROOT_FOLDER_ID\n    datasets: tuple[str, ...] = DEFAULT_DATASETS\n    folder_map_path: Path = Path("automation/drive_dataset_folder_map_20260901.csv")\n    catalog_manifest_path: Path = Path("automation/dataset_archive_manifest_20260901.csv")\n    state_file_name: str = "dataset_transfer_state_20260901.json"\n    plan_output_path: Path | None = Path("/content/dataset_transfer_plan.csv")\n    allow_license_hold_archive: bool = False\n\n    def validate(self) -> None:\n        if self.max_run_bytes <= 0:\n            raise ValueError("max_run_bytes must be positive")\n        if self.chunk_bytes <= 0 or self.chunk_bytes % (256 * 1024) != 0:\n            raise ValueError("chunk_bytes must be a positive multiple of 256 KiB")\n        if self.state_flush_bytes < self.chunk_bytes:\n            raise ValueError("state_flush_bytes must be at least one upload chunk")\n        if not self.target_root_folder_id.strip():\n            raise ValueError("target_root_folder_id is required")\n        unknown = set(self.datasets) - set(SUPPORTED_DATASETS)\n        if unknown:\n            raise ValueError(f"unsupported datasets: {sorted(unknown)}")\n\n\ndef _hf_tree(repo_id: str, revision: str) -> list[dict[str, Any]]:\n    repo = quote(repo_id, safe="/")\n    rev = quote(revision, safe="")\n    next_url: str | None = (\n        f"https://huggingface.co/api/datasets/{repo}/tree/{rev}"\n        "?recursive=true&expand=true"\n    )\n    rows: list[dict[str, Any]] = []\n    while next_url:\n        response = request_with_retry("GET", next_url)\n        response.raise_for_status()\n        page = response.json()\n        if not isinstance(page, list):\n            raise RuntimeError(f"unexpected Hugging Face tree response for {repo_id}")\n        rows.extend(item for item in page if item.get("type") == "file")\n        next_url = response.links.get("next", {}).get("url")\n    return rows\n\n\ndef _hf_source(\n    *,\n    dataset: str,\n    repo_id: str,\n    revision: str,\n    include: Callable[[str], bool],\n    license_name: str,\n) -> list[SourceFile]:\n    files: list[SourceFile] = []\n    for item in _hf_tree(repo_id, revision):\n        path = str(item["path"])\n        if not include(path):\n            continue\n        lfs = item.get("lfs") or {}\n        checksum_algorithm = "sha256" if lfs.get("oid") else "git-oid"\n        checksum_value = str(lfs.get("oid") or item.get("oid") or "") or None\n        files.append(\n            SourceFile(\n                dataset=dataset,\n                relative_path=path,\n                url=(\n                    f"https://huggingface.co/datasets/{quote(repo_id, safe=\'/\')}/resolve/"\n                    f"{quote(revision, safe=\'\')}/{quote(path, safe=\'/\')}?download=true"\n                ),\n                size=int(item["size"]),\n                source_revision=revision,\n                license_name=license_name,\n                license_url=(\n                    f"https://huggingface.co/datasets/{quote(repo_id, safe=\'/\')}/blob/"\n                    f"{quote(revision, safe=\'\')}/README.md"\n                ),\n                checksum_algorithm=checksum_algorithm,\n                checksum_value=checksum_value,\n                source_page=f"https://huggingface.co/datasets/{repo_id}",\n            )\n        )\n    if not files:\n        raise RuntimeError(f"no selected files found in Hugging Face dataset {repo_id}")\n    return sorted(\n        files,\n        key=lambda item: (\n            0 if item.relative_path.lower() in {"readme", "readme.md", "license", "license.txt"} else 1,\n            item.relative_path,\n        ),\n    )\n\n\ndef enumerate_mixfake() -> list[SourceFile]:\n    return _hf_source(\n        dataset="MixFake",\n        repo_id="Tnxts/MixFake",\n        revision=HF_REVISIONS["MixFake"],\n        include=lambda path: bool(re.fullmatch(r"MixFake\\.7z\\.\\d{3}", path))\n        or path == "README.md",\n        license_name="CC BY 4.0",\n    )\n\n\ndef enumerate_dfadd() -> list[SourceFile]:\n    # Archive exports are the canonical downloadable audio.  Parquet mirrors and\n    # the unrelated PFlowTTS checkpoint are deliberately excluded to avoid\n    # duplicates and model-weight contamination in the training-data archive.\n    return _hf_source(\n        dataset="DFADD",\n        repo_id="isjwdu/DFADD",\n        revision=HF_REVISIONS["DFADD"],\n        include=lambda path: bool(re.fullmatch(r"DATASET_[^/]+\\.zip", path))\n        or path == "README.md",\n        license_name="MIT plus source-dataset licences",\n    )\n\n\ndef enumerate_asvspoof5() -> list[SourceFile]:\n    record_id = "14498691"\n    api_url = f"https://zenodo.org/api/records/{record_id}"\n    response = request_with_retry("GET", api_url)\n    response.raise_for_status()\n    payload = response.json()\n    revision = str(payload.get("doi") or f"zenodo:{record_id}")\n    selected: list[SourceFile] = []\n    for item in payload.get("files", []):\n        name = str(item["key"])\n        if not (\n            name.startswith("flac_")\n            or name in {"ASVspoof5_protocols.tar.gz", "README.txt", "LICENSE.txt"}\n        ):\n            continue\n        checksum_text = str(item.get("checksum") or "")\n        algorithm, _, value = checksum_text.partition(":")\n        selected.append(\n            SourceFile(\n                dataset="ASVspoof5",\n                relative_path=name,\n                url=str(item["links"]["self"]),\n                size=int(item["size"]),\n                source_revision=revision,\n                license_name="ODC-By; bona fide speech CC BY 4.0",\n                license_url=f"https://zenodo.org/records/{record_id}",\n                checksum_algorithm=algorithm or None,\n                checksum_value=value or None,\n                source_page=f"https://zenodo.org/records/{record_id}",\n            )\n        )\n\n    def split_order(source: SourceFile) -> tuple[int, str]:\n        name = source.name\n        if name in {"README.txt", "LICENSE.txt", "ASVspoof5_protocols.tar.gz"}:\n            return (0, name)\n        if name.startswith("flac_T_"):\n            return (1, name)\n        if name.startswith("flac_D_"):\n            return (2, name)\n        return (3, name)\n\n    if not selected:\n        raise RuntimeError("Zenodo ASVspoof 5 record contains no selected files")\n    return sorted(selected, key=split_order)\n\n\ndef _head_size(url: str) -> int:\n    response = request_with_retry("HEAD", url, allow_redirects=True)\n    response.raise_for_status()\n    length = response.headers.get("Content-Length")\n    if length is None:\n        probe = request_with_retry(\n            "GET",\n            url,\n            headers={"Range": "bytes=0-0", "Accept-Encoding": "identity"},\n            stream=True,\n            allow_redirects=True,\n        )\n        probe.raise_for_status()\n        content_range = probe.headers.get("Content-Range", "")\n        probe.close()\n        match = re.search(r"/(\\d+)$", content_range)\n        if not match:\n            raise RuntimeError(f"source did not report size: {url}")\n        return int(match.group(1))\n    return int(length)\n\n\ndef enumerate_fma_small() -> list[SourceFile]:\n    release = "FMA release 2017-05-09"\n    base = "https://os.unil.cloud.switch.ch/fma"\n    entries = (\n        ("fma_metadata.zip", "f0df49ffe5f2a6008d7dc83c6915b31835dfe733"),\n        ("fma_small.zip", "ade154f733639d52e35e32f5593efe5be76c6d70"),\n    )\n    return [\n        SourceFile(\n            dataset="FMA_small",\n            relative_path=name,\n            url=f"{base}/{name}",\n            size=_head_size(f"{base}/{name}"),\n            source_revision=release,\n            license_name="Per-track Creative Commons; see fma_metadata.zip",\n            license_url="https://github.com/mdeff/fma/blob/master/README.md",\n            checksum_algorithm="sha1",\n            checksum_value=sha1,\n            source_page="https://github.com/mdeff/fma",\n        )\n        for name, sha1 in entries\n    ]\n\n\ndef _zenodo_source(\n    *,\n    dataset: str,\n    record_id: str,\n    license_name: str,\n    include: Callable[[str], bool] | None = None,\n) -> list[SourceFile]:\n    """Enumerate immutable files from one public Zenodo record."""\n    api_url = f"https://zenodo.org/api/records/{record_id}"\n    response = request_with_retry("GET", api_url)\n    response.raise_for_status()\n    payload = response.json()\n    revision = str(payload.get("doi") or f"zenodo:{record_id}")\n    sources: list[SourceFile] = []\n    for item in payload.get("files", []):\n        name = str(item["key"])\n        if include is not None and not include(name):\n            continue\n        checksum_text = str(item.get("checksum") or "")\n        algorithm, _, value = checksum_text.partition(":")\n        sources.append(\n            SourceFile(\n                dataset=dataset,\n                relative_path=name,\n                url=str(item["links"]["self"]),\n                size=int(item["size"]),\n                source_revision=revision,\n                license_name=license_name,\n                license_url=f"https://zenodo.org/records/{record_id}",\n                checksum_algorithm=algorithm or None,\n                checksum_value=value or None,\n                source_page=f"https://zenodo.org/records/{record_id}",\n            )\n        )\n    if not sources:\n        raise RuntimeError(f"Zenodo record {record_id} contains no selected files")\n    return sorted(sources, key=lambda item: item.relative_path)\n\n\ndef enumerate_wavefake() -> list[SourceFile]:\n    return _zenodo_source(\n        dataset="WaveFake_v1.2",\n        record_id="5642694",\n        license_name="CC BY-SA 4.0",\n    )\n\n\ndef enumerate_fakemusiccaps() -> list[SourceFile]:\n    # Archived under 03_license_hold.  The record currently declares\n    # CC BY-NC 4.0, so this archive must not enter DACON training until the\n    # organiser confirms that non-commercial data are acceptable.\n    return _zenodo_source(\n        dataset="FakeMusicCaps_v2",\n        record_id="15063698",\n        license_name="CC BY-NC 4.0 (archive only; organiser clearance required)",\n    )\n\n\ndef enumerate_speechfake() -> list[SourceFile]:\n    """Enumerate the pinned public SpeechFake ModelScope repository."""\n    dataset = "SpeechFake"\n    repo_id = "inclusionAI/SPEECHFAKE"\n    revision = MODELSCOPE_REVISIONS[dataset]\n    endpoint = f"https://modelscope.cn/api/v1/datasets/{repo_id}"\n    response = request_with_retry(\n        "GET",\n        f"{endpoint}/repo/tree",\n        params={\n            "Revision": revision,\n            "Recursive": "True",\n            "PageNumber": 1,\n            "PageSize": 200,\n        },\n    )\n    response.raise_for_status()\n    payload = response.json()\n    if int(payload.get("Code", 0)) != 200:\n        raise RuntimeError(f"ModelScope SpeechFake listing failed: {payload}")\n    rows = (payload.get("Data") or {}).get("Files") or []\n    sources: list[SourceFile] = []\n    for item in rows:\n        if str(item.get("Type", "")).lower() != "blob":\n            continue\n        path = str(item["Path"])\n        sha256 = str(item.get("Sha256") or "") or None\n        sources.append(\n            SourceFile(\n                dataset=dataset,\n                relative_path=path,\n                url=(\n                    f"{endpoint}/repo?Revision={quote(revision, safe=\'\')}"\n                    f"&FilePath={quote(path, safe=\'\')}"\n                ),\n                size=int(item["Size"]),\n                source_revision=revision,\n                license_name="CC BY 4.0 plus source/generator licences",\n                license_url="https://github.com/YMLLG/SpeechFake/blob/main/LICENSE",\n                checksum_algorithm="sha256" if sha256 else None,\n                checksum_value=sha256,\n                source_page="https://modelscope.cn/datasets/inclusionAI/SPEECHFAKE",\n            )\n        )\n    if not sources:\n        raise RuntimeError("ModelScope SpeechFake repository contains no files")\n    return sorted(\n        sources,\n        key=lambda item: (\n            0 if item.relative_path in {"README.md", "LICENSE.txt", "metadata/metadata.zip"} else 1,\n            item.relative_path,\n        ),\n    )\n\n\ndef enumerate_sonics() -> list[SourceFile]:\n    return _hf_source(\n        dataset="SONICS",\n        repo_id="awsaf49/sonics",\n        revision=HF_REVISIONS["SONICS"],\n        include=lambda path: True,\n        license_name="CC BY-NC 4.0 (archive only; organiser clearance required)",\n    )\n\n\nSOURCE_ENUMERATORS: Mapping[str, Callable[[], list[SourceFile]]] = {\n    "MixFake": enumerate_mixfake,\n    "DFADD": enumerate_dfadd,\n    "ASVspoof5": enumerate_asvspoof5,\n    "FMA_small": enumerate_fma_small,\n    "SpeechFake": enumerate_speechfake,\n    "WaveFake_v1.2": enumerate_wavefake,\n    "SONICS": enumerate_sonics,\n    "FakeMusicCaps_v2": enumerate_fakemusiccaps,\n}\n\n\ndef enumerate_sources(datasets: Sequence[str]) -> list[SourceFile]:\n    sources: list[SourceFile] = []\n    for dataset in datasets:\n        print(f"Discovering {dataset} source files...")\n        current = SOURCE_ENUMERATORS[dataset]()\n        print(\n            f"  {len(current)} files / "\n            f"{human_bytes(sum(item.size for item in current))}"\n        )\n        sources.extend(current)\n    return sources\n\n\ndef load_folder_map(path: Path) -> dict[str, str]:\n    with path.open("r", encoding="utf-8-sig", newline="") as handle:\n        rows = csv.DictReader(handle)\n        mapping = {\n            canonical_dataset_name(row["dataset"]): row["drive_folder_id"].strip()\n            for row in rows\n        }\n    return mapping\n\n\ndef validate_catalog(\n    path: Path,\n    datasets: Sequence[str],\n    *,\n    allow_license_hold_archive: bool = False,\n) -> None:\n    with path.open("r", encoding="utf-8-sig", newline="") as handle:\n        rows = list(csv.DictReader(handle))\n    allowed_classes = {"training_clear"}\n    if allow_license_hold_archive:\n        allowed_classes.add("license_hold")\n    approved = {\n        canonical_dataset_name(row["dataset"])\n        for row in rows\n        if row["use_class"] in allowed_classes and row["priority"] != "BLOCK"\n    }\n    missing = set(datasets) - approved\n    if missing:\n        raise ValueError(\n            "dataset is not permitted by the archive manifest/config: "\n            f"{sorted(missing)}"\n        )\n\n\nclass DriveClient:\n    """Minimal Google Drive wrapper plus remote-source resumable uploader."""\n\n    def __init__(self, credentials: Any):\n        from google.auth.transport.requests import Request as GoogleAuthRequest\n        from googleapiclient.discovery import build\n\n        self.credentials = credentials\n        self.google_auth_request = GoogleAuthRequest()\n        self.service = build("drive", "v3", credentials=credentials, cache_discovery=False)\n        self.http = requests.Session()\n\n    def _auth_headers(self) -> dict[str, str]:\n        if not self.credentials.valid:\n            self.credentials.refresh(self.google_auth_request)\n        return {"Authorization": f"Bearer {self.credentials.token}"}\n\n    def metadata(self, file_id: str) -> dict[str, Any]:\n        return (\n            self.service.files()\n            .get(\n                fileId=file_id,\n                fields="id,name,mimeType,parents,driveId,trashed",\n                supportsAllDrives=True,\n            )\n            .execute()\n        )\n\n    def is_descendant_of(self, folder_id: str, root_id: str) -> bool:\n        pending = [folder_id]\n        visited: set[str] = set()\n        for _ in range(20):\n            if not pending:\n                return False\n            current = pending.pop()\n            if current == root_id:\n                return True\n            if current in visited:\n                continue\n            visited.add(current)\n            pending.extend(self.metadata(current).get("parents", []))\n        return False\n\n    def verify_folder_tree(self, root_id: str, folder_map: Mapping[str, str]) -> bool:\n        root = self.metadata(root_id)\n        if root.get("mimeType") != DRIVE_FOLDER_MIME or root.get("trashed"):\n            raise RuntimeError(f"target root is not an active Drive folder: {root_id}")\n        for dataset, folder_id in folder_map.items():\n            meta = self.metadata(folder_id)\n            if meta.get("mimeType") != DRIVE_FOLDER_MIME or meta.get("trashed"):\n                raise RuntimeError(f"{dataset} destination is not an active folder: {folder_id}")\n            if not self.is_descendant_of(folder_id, root_id):\n                raise RuntimeError(\n                    f"{dataset} folder {folder_id} is not inside target root {root_id}"\n                )\n        return bool(root.get("driveId"))\n\n    def storage_free_bytes(self) -> int | None:\n        quota = self.service.about().get(fields="storageQuota").execute().get("storageQuota", {})\n        limit = quota.get("limit")\n        usage = quota.get("usage")\n        if limit is None or usage is None:\n            return None\n        return max(0, int(limit) - int(usage))\n\n    def list_children(self, parent_id: str) -> dict[str, list[dict[str, Any]]]:\n        escaped = parent_id.replace("\'", "\\\\\'")\n        token: str | None = None\n        result: dict[str, list[dict[str, Any]]] = {}\n        while True:\n            response = (\n                self.service.files()\n                .list(\n                    q=f"\'{escaped}\' in parents and trashed = false",\n                    fields="nextPageToken,files(id,name,size,md5Checksum,mimeType)",\n                    pageSize=1000,\n                    pageToken=token,\n                    supportsAllDrives=True,\n                    includeItemsFromAllDrives=True,\n                    corpora="allDrives",\n                )\n                .execute()\n            )\n            for item in response.get("files", []):\n                result.setdefault(str(item["name"]), []).append(item)\n            token = response.get("nextPageToken")\n            if not token:\n                return result\n\n    def _find_named_file(self, parent_id: str, name: str) -> list[dict[str, Any]]:\n        escaped_parent = parent_id.replace("\'", "\\\\\'")\n        escaped_name = name.replace("\'", "\\\\\'")\n        response = (\n            self.service.files()\n            .list(\n                q=(\n                    f"\'{escaped_parent}\' in parents and name = \'{escaped_name}\' "\n                    "and trashed = false"\n                ),\n                fields="files(id,name,size,modifiedTime)",\n                pageSize=10,\n                supportsAllDrives=True,\n                includeItemsFromAllDrives=True,\n                corpora="allDrives",\n            )\n            .execute()\n        )\n        return list(response.get("files", []))\n\n    def load_state(self, parent_id: str, name: str) -> tuple[dict[str, Any], str | None]:\n        matches = self._find_named_file(parent_id, name)\n        if len(matches) > 1:\n            raise RuntimeError(f"multiple Drive state files named {name!r}")\n        if not matches:\n            return ({"schema_version": 1, "sources": {}, "runs": []}, None)\n        file_id = str(matches[0]["id"])\n        payload = self.service.files().get_media(fileId=file_id).execute()\n        return (json.loads(payload.decode("utf-8")), file_id)\n\n    def save_state(\n        self,\n        parent_id: str,\n        name: str,\n        state: Mapping[str, Any],\n        file_id: str | None,\n    ) -> str:\n        from googleapiclient.errors import HttpError\n        from googleapiclient.http import MediaIoBaseUpload\n\n        payload = json.dumps(state, ensure_ascii=False, indent=2, sort_keys=True).encode("utf-8")\n        last_error: Exception | None = None\n        # State persistence is called repeatedly during multi-GB transfers.  A\n        # stale keep-alive socket can fail with BrokenPipeError even while the\n        # resumable upload session itself remains healthy.  Rebuild both the\n        # media object and request on every attempt so no consumed BytesIO is\n        # reused after a transport failure.\n        for attempt in range(6):\n            media = MediaIoBaseUpload(io.BytesIO(payload), mimetype=STATE_MIME, resumable=False)\n            if file_id:\n                request = self.service.files().update(\n                    fileId=file_id,\n                    media_body=media,\n                    fields="id",\n                    supportsAllDrives=True,\n                )\n            else:\n                request = self.service.files().create(\n                    body={"name": name, "parents": [parent_id], "mimeType": STATE_MIME},\n                    media_body=media,\n                    fields="id",\n                    supportsAllDrives=True,\n                )\n            try:\n                response = request.execute(num_retries=3)\n                return str(response["id"])\n            except (OSError, HttpError, Urllib3HTTPError) as exc:\n                last_error = exc\n                if attempt + 1 >= 6:\n                    break\n                print(\n                    f"    transient Drive state-save error; retrying "\n                    f"({attempt + 1}/6): {exc}"\n                )\n                time.sleep(min(2**attempt, 20))\n        raise RuntimeError("Drive state save failed after 6 attempts") from last_error\n\n    def begin_resumable_upload(self, source: SourceFile, parent_id: str) -> str:\n        endpoint = (\n            "https://www.googleapis.com/upload/drive/v3/files"\n            "?uploadType=resumable&supportsAllDrives=true"\n            "&fields=id,name,size,md5Checksum"\n        )\n        metadata = {\n            "name": source.name,\n            "parents": [parent_id],\n            "description": (\n                f"Deepvoice dataset archive\\nsource={source.url}\\n"\n                f"revision={source.source_revision}\\nlicense={source.license_name}\\n"\n                f"checksum={source.checksum or \'not-declared\'}"\n            ),\n        }\n        headers = self._auth_headers()\n        headers.update(\n            {\n                "Content-Type": "application/json; charset=UTF-8",\n                "X-Upload-Content-Type": "application/octet-stream",\n                "X-Upload-Content-Length": str(source.size),\n            }\n        )\n        response = request_with_retry(\n            "POST",\n            endpoint,\n            session=self.http,\n            headers=headers,\n            data=json.dumps(metadata).encode("utf-8"),\n        )\n        response.raise_for_status()\n        location = response.headers.get("Location")\n        if not location:\n            raise RuntimeError("Google Drive did not return a resumable upload URL")\n        return location\n\n    def query_resumable_offset(self, session_url: str, total: int) -> tuple[int, dict[str, Any] | None]:\n        headers = self._auth_headers()\n        headers.update({"Content-Length": "0", "Content-Range": f"bytes */{total}"})\n        response = self.http.put(session_url, headers=headers, timeout=(30, 180))\n        if response.status_code in {200, 201}:\n            return total, response.json()\n        if response.status_code == 308:\n            match = re.search(r"bytes=0-(\\d+)", response.headers.get("Range", ""))\n            return (int(match.group(1)) + 1 if match else 0), None\n        if response.status_code in {404, 410}:\n            raise FileNotFoundError("resumable upload session expired")\n        response.raise_for_status()\n        raise AssertionError("unreachable")\n\n    def put_chunk(\n        self,\n        session_url: str,\n        *,\n        start: int,\n        data: bytes,\n        total: int,\n    ) -> tuple[int, dict[str, Any] | None]:\n        end = start + len(data) - 1\n        headers = self._auth_headers()\n        headers.update(\n            {\n                "Content-Type": "application/octet-stream",\n                "Content-Length": str(len(data)),\n                "Content-Range": f"bytes {start}-{end}/{total}",\n            }\n        )\n        response = self.http.put(session_url, headers=headers, data=data, timeout=(30, 300))\n        if response.status_code in {200, 201}:\n            return total, response.json()\n        if response.status_code == 308:\n            match = re.search(r"bytes=0-(\\d+)", response.headers.get("Range", ""))\n            return (int(match.group(1)) + 1 if match else 0), None\n        if response.status_code in {404, 410}:\n            raise FileNotFoundError("resumable upload session expired")\n        response.raise_for_status()\n        raise AssertionError("unreachable")\n\n    def completed_file_metadata(self, file_id: str) -> dict[str, Any]:\n        return (\n            self.service.files()\n            .get(\n                fileId=file_id,\n                fields="id,name,size,md5Checksum,createdTime",\n                supportsAllDrives=True,\n            )\n            .execute()\n        )\n\n\ndef _read_at_most(raw: Any, target: int) -> bytes:\n    buffer = bytearray()\n    while len(buffer) < target:\n        chunk = raw.read(target - len(buffer))\n        if not chunk:\n            break\n        buffer.extend(chunk)\n    return bytes(buffer)\n\n\ndef transfer_one(\n    *,\n    drive: DriveClient,\n    source: SourceFile,\n    destination_folder_id: str,\n    state: dict[str, Any],\n    save_state: Callable[[], None],\n    config: TransferConfig,\n) -> dict[str, Any]:\n    record = state["sources"].setdefault(source.key, source.manifest_record())\n    record.update(source.manifest_record())\n    record["destination_folder_id"] = destination_folder_id\n    record.setdefault("status", "pending")\n    record.setdefault("uploaded_bytes", 0)\n    session_url = record.get("upload_session_url")\n\n    completed_payload: dict[str, Any] | None = None\n    if session_url:\n        try:\n            offset, completed_payload = drive.query_resumable_offset(session_url, source.size)\n        except FileNotFoundError:\n            session_url = None\n            offset = 0\n            record.update({"upload_session_url": None, "uploaded_bytes": 0})\n    else:\n        offset = 0\n\n    if completed_payload is not None:\n        offset = source.size\n    if offset < source.size and not session_url:\n        session_url = drive.begin_resumable_upload(source, destination_folder_id)\n        record.update(\n            {\n                "status": "uploading",\n                "upload_session_url": session_url,\n                "uploaded_bytes": 0,\n                "started_at": utc_now(),\n            }\n        )\n        save_state()\n\n    initial_offset = offset\n    full_hasher = (\n        hashlib.new(source.checksum_algorithm)\n        if initial_offset == 0 and source.checksum_algorithm in hashlib.algorithms_available\n        else None\n    )\n    bytes_since_flush = 0\n    retry_count = 0\n\n    while offset < source.size:\n        headers = {"Accept-Encoding": "identity"}\n        if offset:\n            headers["Range"] = f"bytes={offset}-"\n        try:\n            response = request_with_retry(\n                "GET",\n                source.url,\n                headers=headers,\n                stream=True,\n                allow_redirects=True,\n                attempts=3,\n                timeout=(30, 300),\n            )\n            if offset and response.status_code != 206:\n                response.close()\n                raise RuntimeError(\n                    f"source does not support byte-range resume at offset {offset}: {source.url}"\n                )\n            response.raise_for_status()\n            response.raw.decode_content = False\n            while offset < source.size:\n                wanted = min(config.chunk_bytes, source.size - offset)\n                data = _read_at_most(response.raw, wanted)\n                if len(data) != wanted:\n                    raise IOError(\n                        f"source ended early at {offset + len(data)} / {source.size}: {source.url}"\n                    )\n                chunk_start = offset\n                next_offset, completed_payload = drive.put_chunk(\n                    session_url,\n                    start=chunk_start,\n                    data=data,\n                    total=source.size,\n                )\n                accepted = max(0, min(len(data), next_offset - chunk_start))\n                if accepted and full_hasher is not None:\n                    full_hasher.update(data[:accepted])\n                record.setdefault("chunk_hashes", []).append(\n                    {\n                        "offset": chunk_start,\n                        "length": accepted,\n                        "sha256": hashlib.sha256(data[:accepted]).hexdigest(),\n                    }\n                )\n                offset = next_offset\n                bytes_since_flush += accepted\n                record.update({"uploaded_bytes": offset, "updated_at": utc_now()})\n                print(\n                    f"    {source.name}: {100 * offset / source.size:6.2f}% "\n                    f"({human_bytes(offset)} / {human_bytes(source.size)})"\n                )\n                if bytes_since_flush >= config.state_flush_bytes:\n                    save_state()\n                    bytes_since_flush = 0\n                if accepted != len(data):\n                    break\n            response.close()\n            retry_count = 0\n        # Streaming reads are performed by urllib3 underneath requests.  A\n        # mid-file disconnect can therefore surface as urllib3.ProtocolError\n        # instead of requests.RequestException.  Treat both as resumable: the\n        # Drive session is queried for its accepted offset and the source is\n        # reopened with a Range request from that exact byte.\n        except (requests.RequestException, Urllib3HTTPError, OSError, RuntimeError) as exc:\n            retry_count += 1\n            if retry_count > 5:\n                record.update({"status": "error", "error": repr(exc), "updated_at": utc_now()})\n                save_state()\n                raise\n            print(f"    transient transfer error; resuming ({retry_count}/5): {exc}")\n            time.sleep(min(2**retry_count, 20))\n            previous_offset = offset\n            offset, completed_payload = drive.query_resumable_offset(session_url, source.size)\n            if offset != previous_offset:\n                # The upload may have accepted bytes even though the client did\n                # not receive the response.  Resume remains safe, but a local\n                # whole-file digest can no longer be proven contiguous.\n                full_hasher = None\n            record.update({"uploaded_bytes": offset, "updated_at": utc_now()})\n            save_state()\n\n    if completed_payload is None:\n        offset, completed_payload = drive.query_resumable_offset(session_url, source.size)\n    if offset != source.size or not completed_payload:\n        raise RuntimeError(f"Drive did not finalize {source.name}")\n    file_id = str(completed_payload["id"])\n    metadata = drive.completed_file_metadata(file_id)\n    if int(metadata.get("size", -1)) != source.size:\n        raise RuntimeError(\n            f"Drive size mismatch for {source.name}: {metadata.get(\'size\')} != {source.size}"\n        )\n\n    observed_checksum: str | None = None\n    checksum_verified: bool | None = None\n    if full_hasher is not None and initial_offset == 0:\n        observed_checksum = full_hasher.hexdigest()\n        if source.checksum_value:\n            checksum_verified = observed_checksum.lower() == source.checksum_value.lower()\n    elif source.checksum_algorithm == "md5" and metadata.get("md5Checksum"):\n        observed_checksum = str(metadata["md5Checksum"])\n        checksum_verified = observed_checksum.lower() == str(source.checksum_value).lower()\n\n    record.update(\n        {\n            "status": "complete",\n            "drive_file_id": file_id,\n            "drive_size": int(metadata["size"]),\n            "drive_md5": metadata.get("md5Checksum"),\n            "observed_checksum": observed_checksum,\n            "checksum_verified": checksum_verified,\n            "uploaded_bytes": source.size,\n            "completed_at": utc_now(),\n            "updated_at": utc_now(),\n            "upload_session_url": None,\n            "error": None,\n        }\n    )\n    save_state()\n    if checksum_verified is False:\n        raise RuntimeError(f"checksum mismatch after upload: {source.name}")\n    return record\n\n\ndef build_plan(\n    sources: Sequence[SourceFile],\n    *,\n    max_bytes: int,\n    existing_by_dataset: Mapping[str, Mapping[str, list[dict[str, Any]]]] | None = None,\n    state: Mapping[str, Any] | None = None,\n) -> tuple[list[dict[str, Any]], list[SourceFile]]:\n    plan_rows: list[dict[str, Any]] = []\n    selected: list[SourceFile] = []\n    budget_used = 0\n    source_state = (state or {}).get("sources", {})\n\n    for source in sources:\n        action = "upload"\n        reason = "missing"\n        remaining = source.size\n        budget_bytes = source.size\n        existing = list((existing_by_dataset or {}).get(source.dataset, {}).get(source.name, []))\n        if len(existing) > 1:\n            raise RuntimeError(\n                f"duplicate destination name in {source.dataset}: {source.name}"\n            )\n        if existing:\n            drive_size = int(existing[0].get("size", -1))\n            if drive_size == source.size:\n                action, reason, remaining, budget_bytes = (\n                    "skip",\n                    "same name and size already in Drive",\n                    0,\n                    0,\n                )\n            else:\n                raise RuntimeError(\n                    f"Drive name collision with different size: {source.dataset}/{source.name} "\n                    f"({drive_size} != {source.size}); no overwrite performed"\n                )\n        elif source_state.get(source.key, {}).get("status") == "complete":\n            # Drive listing is authoritative.  A state-only completion is not\n            # trusted because the file may have been moved outside the folder.\n            reason = "state says complete but destination file is absent; re-upload"\n        else:\n            uploaded = int(source_state.get(source.key, {}).get("uploaded_bytes", 0) or 0)\n            remaining = max(0, source.size - uploaded)\n            if uploaded:\n                reason = f"resume from {human_bytes(uploaded)}"\n                # Charge the full file against the planning limit.  This is\n                # deliberately conservative: a Drive resumable session can\n                # expire between planning and upload, in which case the source\n                # must restart from byte zero without exceeding the run budget.\n                budget_bytes = source.size\n\n        if action == "upload":\n            if budget_bytes <= max_bytes - budget_used:\n                selected.append(source)\n                budget_used += budget_bytes\n            else:\n                action, reason = "defer", "per-run byte budget"\n\n        plan_rows.append(\n            {\n                **source.manifest_record(),\n                "action": action,\n                "reason": reason,\n                "remaining_bytes": remaining,\n                "budget_bytes": budget_bytes,\n                "planned_this_run": source in selected,\n            }\n        )\n    return plan_rows, selected\n\n\ndef write_plan(path: Path | None, rows: Sequence[Mapping[str, Any]]) -> None:\n    if path is None or not rows:\n        return\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))\n        writer.writeheader()\n        writer.writerows(rows)\n    print(f"Plan CSV: {path}")\n\n\ndef authenticate_colab_drive() -> Any:\n    try:\n        from google.colab import auth\n        import google.auth\n    except ImportError as exc:\n        raise RuntimeError("actual transfer mode must run in Google Colab") from exc\n    auth.authenticate_user()\n    credentials, _ = google.auth.default(\n        scopes=["https://www.googleapis.com/auth/drive"]\n    )\n    return credentials\n\n\ndef run(config: TransferConfig) -> dict[str, Any]:\n    config.validate()\n    folder_map = load_folder_map(config.folder_map_path)\n    validate_catalog(\n        config.catalog_manifest_path,\n        config.datasets,\n        allow_license_hold_archive=config.allow_license_hold_archive,\n    )\n    missing_folders = set(config.datasets) - set(folder_map)\n    if missing_folders:\n        raise ValueError(f"dataset folders missing from folder map: {sorted(missing_folders)}")\n\n    print("=" * 72)\n    print("Deepvoice dataset archive: public source -> Google Drive")\n    print(f"Mode: {\'DRY RUN (no Drive write)\' if config.dry_run else \'TRANSFER\'}")\n    print(\n        f"Per-run byte budget: {config.max_run_bytes / GB:,.2f} GB "\n        f"({human_bytes(config.max_run_bytes)})"\n    )\n    print(f"Datasets: {\', \'.join(config.datasets)}")\n    print("=" * 72)\n\n    sources = enumerate_sources(config.datasets)\n    drive: DriveClient | None = None\n    state: dict[str, Any] = {"schema_version": 1, "sources": {}, "runs": []}\n    state_file_id: str | None = None\n    existing_by_dataset: dict[str, dict[str, list[dict[str, Any]]]] = {}\n    is_shared_drive = False\n\n    if not config.dry_run:\n        drive = DriveClient(authenticate_colab_drive())\n        selected_folder_map = {dataset: folder_map[dataset] for dataset in config.datasets}\n        is_shared_drive = drive.verify_folder_tree(\n            config.target_root_folder_id, selected_folder_map\n        )\n        state, state_file_id = drive.load_state(\n            config.target_root_folder_id, config.state_file_name\n        )\n        for dataset in config.datasets:\n            existing_by_dataset[dataset] = drive.list_children(folder_map[dataset])\n\n    plan_rows, selected = build_plan(\n        sources,\n        max_bytes=config.max_run_bytes,\n        existing_by_dataset=existing_by_dataset if drive else None,\n        state=state,\n    )\n    write_plan(config.plan_output_path, plan_rows)\n\n    planned_bytes = sum(\n        int(row["budget_bytes"]) for row in plan_rows if row["planned_this_run"]\n    )\n    print(\n        f"Selected: {len(selected)} / {len(sources)} files, "\n        f"{human_bytes(planned_bytes)} this run"\n    )\n    for row in plan_rows:\n        marker = "UPLOAD" if row["planned_this_run"] else str(row["action"]).upper()\n        print(\n            f"  [{marker:6}] {row[\'dataset\']}/{row[\'name\']} "\n            f"{human_bytes(int(row[\'size\']))} - {row[\'reason\']}"\n        )\n\n    summary = {\n        "dry_run": config.dry_run,\n        "datasets": list(config.datasets),\n        "source_file_count": len(sources),\n        "source_total_bytes": sum(source.size for source in sources),\n        "selected_file_count": len(selected),\n        "planned_bytes": planned_bytes,\n        "max_run_bytes": config.max_run_bytes,\n        "completed": [],\n        "started_at": utc_now(),\n    }\n    if config.dry_run:\n        print("\\nDRY_RUN=True: source files were only inventoried; no authentication or upload occurred.")\n        return summary\n\n    assert drive is not None\n    if not is_shared_drive:\n        free_bytes = drive.storage_free_bytes()\n        print(f"Regular My Drive free space: {human_bytes(free_bytes)}")\n        if free_bytes is not None and planned_bytes + config.free_space_reserve_bytes > free_bytes:\n            raise RuntimeError(\n                "transfer stopped before upload: planned bytes plus the 5 GiB reserve exceed "\n                f"regular My Drive free space ({human_bytes(free_bytes)})"\n            )\n    else:\n        print("Target is a Shared Drive; regular My Drive quota check is not applicable.")\n\n    run_record = {\n        "started_at": summary["started_at"],\n        "status": "running",\n        "target_root_folder_id": config.target_root_folder_id,\n        "datasets": list(config.datasets),\n        "max_run_bytes": config.max_run_bytes,\n        "chunk_bytes": config.chunk_bytes,\n        "planned_bytes": planned_bytes,\n        "selected_keys": [source.key for source in selected],\n    }\n    state.setdefault("runs", []).append(run_record)\n\n    def persist_state() -> None:\n        nonlocal state_file_id\n        state["updated_at"] = utc_now()\n        state_file_id = drive.save_state(\n            config.target_root_folder_id,\n            config.state_file_name,\n            state,\n            state_file_id,\n        )\n\n    persist_state()\n    try:\n        for index, source in enumerate(selected, start=1):\n            print(\n                f"\\n[{index}/{len(selected)}] {source.dataset}/{source.name} "\n                f"({human_bytes(source.size)})"\n            )\n            record = transfer_one(\n                drive=drive,\n                source=source,\n                destination_folder_id=folder_map[source.dataset],\n                state=state,\n                save_state=persist_state,\n                config=config,\n            )\n            summary["completed"].append(\n                {\n                    "key": source.key,\n                    "size": source.size,\n                    "drive_file_id": record["drive_file_id"],\n                }\n            )\n        run_record.update({"status": "complete", "completed_at": utc_now()})\n    except BaseException as exc:\n        run_record.update({"status": "interrupted", "error": repr(exc), "updated_at": utc_now()})\n        persist_state()\n        raise\n    persist_state()\n    complete_keys = {\n        str(row["key"])\n        for row in plan_rows\n        if row["action"] == "skip" or row["planned_this_run"]\n    }\n    complete_bytes = sum(source.size for source in sources if source.key in complete_keys)\n    summary.update(\n        {\n            "complete_file_count_after_run": len(complete_keys),\n            "complete_bytes_after_run": complete_bytes,\n            "remaining_file_count_after_run": len(sources) - len(complete_keys),\n            "remaining_bytes_after_run": sum(source.size for source in sources) - complete_bytes,\n        }\n    )\n    summary["completed_at"] = utc_now()\n    print("\\nRun complete. Re-run the same notebook to continue with deferred files.")\n    return summary\n\n\ndef parse_args(argv: Sequence[str] | None = None) -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--execute", action="store_true", help="perform Drive writes (Colab only)")\n    parser.add_argument("--budget-gb", type=float, default=40.0)\n    parser.add_argument("--chunk-mib", type=int, default=16)\n    parser.add_argument("--root-folder-id", default=DEFAULT_ROOT_FOLDER_ID)\n    parser.add_argument("--folder-map", type=Path, default=TransferConfig.folder_map_path)\n    parser.add_argument("--catalog", type=Path, default=TransferConfig.catalog_manifest_path)\n    parser.add_argument("--datasets", nargs="+", default=list(DEFAULT_DATASETS))\n    parser.add_argument("--plan-output", type=Path, default=Path("/content/dataset_transfer_plan.csv"))\n    return parser.parse_args(argv)\n\n\ndef cli(argv: Sequence[str] | None = None) -> dict[str, Any]:\n    args = parse_args(argv)\n    datasets = tuple(canonical_dataset_name(item) for item in args.datasets)\n    config = TransferConfig(\n        dry_run=not args.execute,\n        max_run_bytes=int(args.budget_gb * GB),\n        chunk_bytes=args.chunk_mib * MIB,\n        target_root_folder_id=args.root_folder_id,\n        datasets=datasets,\n        folder_map_path=args.folder_map,\n        catalog_manifest_path=args.catalog,\n        plan_output_path=args.plan_output,\n    )\n    return run(config)\n\n\nif __name__ == "__main__":\n    cli()\n', encoding="utf-8")
FOLDER_MAP_PATH.write_text('dataset,drive_folder_id,drive_folder_url,use_class\nMixFake,1Qq77WhA-RsJ91XnwP5dtFHNFzMbdjXUY,https://drive.google.com/drive/folders/1Qq77WhA-RsJ91XnwP5dtFHNFzMbdjXUY,training_clear\nDFADD,1vSAWTe6UAqfFHRDH6a15sx_RGmQusKUL,https://drive.google.com/drive/folders/1vSAWTe6UAqfFHRDH6a15sx_RGmQusKUL,training_clear\nASVspoof5,1kituXbGaD7w0afOtSJTM7NPRp2BSGgcq,https://drive.google.com/drive/folders/1kituXbGaD7w0afOtSJTM7NPRp2BSGgcq,training_clear\nSpeechFake,1wII-2sswNxooMgABsfjt0mVWvMGkpPGw,https://drive.google.com/drive/folders/1wII-2sswNxooMgABsfjt0mVWvMGkpPGw,training_clear\nSpoofCeleb,119uQGY6XfNqEfq3fUb2qd2-dXZdNOeVc,https://drive.google.com/drive/folders/119uQGY6XfNqEfq3fUb2qd2-dXZdNOeVc,training_clear\nFake-or-Real,1IRISEW_tN4gtTpf4M_lfuboHAI0gE7sj,https://drive.google.com/drive/folders/1IRISEW_tN4gtTpf4M_lfuboHAI0gE7sj,training_clear\nASVspoof2019_LA,1wRJNRpXwLitHnM68hyKnXf2brd80oMFE,https://drive.google.com/drive/folders/1wRJNRpXwLitHnM68hyKnXf2brd80oMFE,training_clear\nWaveFake_v1.2,1zqdcL6yaO0vhr3GjmLKK97bmeOXecg2y,https://drive.google.com/drive/folders/1zqdcL6yaO0vhr3GjmLKK97bmeOXecg2y,training_clear\nFMA_small,14lo1ts2ks_M6jCV9UK1xyKlFD9Bb_rbI,https://drive.google.com/drive/folders/14lo1ts2ks_M6jCV9UK1xyKlFD9Bb_rbI,training_clear\nIn-the-Wild_Audio_Deepfake,1jg4hdceMpZZHHH2sDFdGi0_-E-wTKj3f,https://drive.google.com/drive/folders/1jg4hdceMpZZHHH2sDFdGi0_-E-wTKj3f,validation_only\nASVspoof2021_DF,1qwiO9lRHRhavYA8Qbu4n-C_WP-RI3BW7,https://drive.google.com/drive/folders/1qwiO9lRHRhavYA8Qbu4n-C_WP-RI3BW7,validation_only\nLlamaPartialSpoof,1rWorBq12i3Gb3rERQk9gaa7zQ91q6T3b,https://drive.google.com/drive/folders/1rWorBq12i3Gb3rERQk9gaa7zQ91q6T3b,validation_only\nDeepfake-Eval-2024,1X-6xo7GYWG5dp2tSsJWTP4tszk1_e6BQ,https://drive.google.com/drive/folders/1X-6xo7GYWG5dp2tSsJWTP4tszk1_e6BQ,validation_only\nSONICS,13hrPqkMM8VicGbISVU_BbJuA1__UBVeY,https://drive.google.com/drive/folders/13hrPqkMM8VicGbISVU_BbJuA1__UBVeY,license_hold\nFakeMusicCaps_v2,1DGGM7lWvW8eBqpZB-huoGrOnORlD5eJL,https://drive.google.com/drive/folders/1DGGM7lWvW8eBqpZB-huoGrOnORlD5eJL,license_hold\nCodecfake,1XErnkGndwQ37ZYl_3D0xQynFmRGhB_us,https://drive.google.com/drive/folders/1XErnkGndwQ37ZYl_3D0xQynFmRGhB_us,license_hold\nMLAAD_v9,1HeuiELLW7ucJ5IMIQoomIk0IKePdysaQ,https://drive.google.com/drive/folders/1HeuiELLW7ucJ5IMIQoomIk0IKePdysaQ,license_hold\nReplayDF,1OhPy__4e4hrfVOzmQILEJBXy2kZplypA,https://drive.google.com/drive/folders/1OhPy__4e4hrfVOzmQILEJBXy2kZplypA,license_hold\nSingFake,1C06CTMnMv2J979a81qW9dK4p9QErh9Tt,https://drive.google.com/drive/folders/1C06CTMnMv2J979a81qW9dK4p9QErh9Tt,license_hold\nEnvSDD,1c0Gz2l8C_IC42U0XjcogUZClHvxpXhPK,https://drive.google.com/drive/folders/1c0Gz2l8C_IC42U0XjcogUZClHvxpXhPK,license_hold\nPhonemeDF,1B0Rv-qEr2hgq05y7_pJB_1tNtqdpJ7HM,https://drive.google.com/drive/folders/1B0Rv-qEr2hgq05y7_pJB_1tNtqdpJ7HM,license_hold\nAT-ADD_Track2,1Q73J9ecpMNgGI0Mtpu_KAztCHjegEljj,https://drive.google.com/drive/folders/1Q73J9ecpMNgGI0Mtpu_KAztCHjegEljj,license_hold\n', encoding="utf-8")
CATALOG_PATH.write_text('priority,dataset,use_class,modality,approx_size_gb,license,source_url,download_auth,drive_status,action\nP0,MixFake,training_clear,voice_music_background,71.6,CC BY 4.0,https://huggingface.co/datasets/Tnxts/MixFake,public HuggingFace,missing,download pinned subset first\nP0,DFADD,training_clear,voice_real_fake,42.7,MIT plus source licenses,https://huggingface.co/datasets/isjwdu/DFADD,public HuggingFace,missing,download corrected 2025-04 archive\nP0,ASVspoof 5,training_clear,voice_real_fake,142.3,ODC-By plus bona fide CC BY 4.0,https://zenodo.org/records/14498691,public Zenodo,missing,download split archives\nP0,SpeechFake,training_clear,voice_multilingual_fake,,CC BY 4.0 plus generator licenses,https://github.com/YMLLG/SpeechFake,ModelScope public,missing,download Korean and target generators first\nP1,SpoofCeleb,training_clear,voice_real_world_fake,134,CC BY 4.0,https://www.jungjee.com/spoofceleb/,gated HuggingFace approval,missing,wait for institutional approval\nP1,Fake-or-Real,training_clear,voice_real_fake,16,LGPL-3.0 listing requires redistribution review,https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset,Kaggle token,missing,download after Kaggle auth\nP1,ASVspoof 2019 LA,training_clear,voice_real_fake,7.12,ODC-By,https://datashare.ed.ac.uk/items/31074a11-b6f6-4e92-a4ad-07093f8c0c45,public DataShare,already in Drive,do not duplicate; preserve link and hash\nP2,WaveFake v1.2,training_clear,voice_fake_only,28.9,CC BY-SA 4.0,https://zenodo.org/records/5642694,public Zenodo,missing,review share-alike impact then download\nV0,In-the-Wild Audio Deepfake,validation_only,voice_real_fake,,Apache-2.0,https://deepfake-demo.aisec.fraunhofer.de/in_the_wild,public official,missing,download validation only\nV0,ASVspoof 2021 DF,validation_only,voice_real_fake,34.5,ODC-By,https://zenodo.org/records/4835108,public Zenodo,missing,download multipart validation source\nV1,LlamaPartialSpoof,validation_only,partial_voice_fake,28,CC BY 4.0,https://zenodo.org/records/14214149,public Zenodo,missing,download validation source\nHOLD,SingFake,license_hold,singing_real_fake,,original music rights unclear,https://singfake.org/,annotation URLs,missing,no bulk use before rights review\nHOLD,Codecfake,license_hold,codec_voice_fake,,CC BY-NC-ND 4.0,https://github.com/xieyuankun/Codecfake,public metadata,missing,do not train before organizer clearance\nHOLD,MLAAD v9,license_hold,multilingual_voice_fake,,CC BY-NC 4.0,https://www.deepfake-total.com/mlaad,gated,missing,do not train before organizer clearance\nHOLD,EnvSDD,license_hold,environmental_sound_fake,,dataset license unclear,https://zenodo.org/records/15220951,public Zenodo,missing,hold until license confirmed\nHOLD,ReplayDF,license_hold,replayed_voice_fake,23.9,CC BY-NC 4.0,https://deepfake-total.com/replaydf,public official,missing,do not train before organizer clearance\nHOLD,PhonemeDF,license_hold,phoneme_voice_fake,82.9,dataset license unclear,https://zenodo.org/records/19522979,public Zenodo,missing,hold until license confirmed\nBLOCK,AT-ADD 2026 Track2,license_hold,challenge_voice_fake,,challenge-only,https://huggingface.co/datasets/xieyuankun/AT-ADD-Track2,HuggingFace,missing,do not use for DACON\nBLOCK,Deepfake-Eval-2024,validation_only,voice_real_fake,,training prohibited,https://huggingface.co/datasets/nuriachandra/Deepfake-Eval-2024,HuggingFace,missing,evaluation only and isolated\nHOLD,FakeMusicCaps v2,license_hold,music_fake,12.9,record license unclear,https://zenodo.org/records/15063698,public Zenodo,ephemeral samples only,hold full training until rights confirmed\nHOLD,SONICS,license_hold,music_real_fake,32.6,CC BY-NC 4.0,https://huggingface.co/datasets/awsaf49/sonics,public HuggingFace,ephemeral samples only,organizer clearance before training use\nP1,FMA small,training_clear,music_real,7,per-track Creative Commons varies,https://os.unil.cloud.switch.ch/fma/fma_small.zip,public direct,ephemeral samples only,download with per-track license manifest\n', encoding="utf-8")

expected = 'f5202f25509f98ed9cbc33a1eb5391fc4c1cc98c75d1b9be431d0e0f3297677b'
observed = hashlib.sha256(SCRIPT_PATH.read_bytes()).hexdigest()
assert observed == expected, (observed, expected)
print("✅ 전송 코드 무결성 확인:", observed)


## 5. 누락 파일 탐색 및 전송

Drive 폴더의 실제 파일을 먼저 조회한다. 이미 완료된 파일은 건너뛰고 누락되었거나
중단된 파일만 선택한다. 일반 My Drive에서는 계획 용량 외에 5 GiB 안전 여유가
없으면 쓰기 전에 중단한다.


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("drive_dataset_streamer", SCRIPT_PATH)
streamer = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = streamer
spec.loader.exec_module(streamer)

batch_summaries = []
for batch_index in range(1, MAX_AUTOMATIC_BATCHES + 1):
    print(f"\n===== Automatic transfer batch {batch_index}/{MAX_AUTOMATIC_BATCHES} =====")
    config = streamer.TransferConfig(
        dry_run=DRY_RUN,
        max_run_bytes=int(PER_RUN_BUDGET_GB * streamer.GB),
        chunk_bytes=int(UPLOAD_CHUNK_MIB * streamer.MIB),
        target_root_folder_id=TARGET_ROOT_FOLDER_ID,
        datasets=tuple(SELECTED_DATASETS),
        folder_map_path=FOLDER_MAP_PATH,
        catalog_manifest_path=CATALOG_PATH,
        state_file_name="dataset_transfer_state_20260901.json",
        plan_output_path=RUNTIME_DIR / "dataset_transfer_plan.csv",
        allow_license_hold_archive=True,
    )
    summary = streamer.run(config)
    batch_summaries.append(summary)
    if int(summary.get("remaining_bytes_after_run", 0)) == 0:
        print("All selected dataset files are present in Drive.")
        break
else:
    print("Automatic batch limit reached. Run All again to continue from Drive state.")
summary


## 6. 실행 결과 저장

자동 실행기가 상태를 판단할 수 있도록 결과 JSON과 마지막 계획 CSV를
`/content/data001`에 저장한다. 런타임이 중단돼도 같은 노트북을 다시 실행하면 된다.


In [ ]:
import json

RESULT_DIR = Path("/content/data001")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_PATH = RESULT_DIR / "exp998_result.json"
PLAN_PATH = RESULT_DIR / "data005_transfer_plan.csv"
PLAN_PATH.write_bytes((RUNTIME_DIR / "dataset_transfer_plan.csv").read_bytes())

payload = {
    "experiment_id": "EXP998",
    "task_id": "DATA001",
    "archive_task_id": "DATA005",
    "status": "completed",
    "summary": summary,
    "artifacts": [str(PLAN_PATH)],
}
RESULT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

print("=" * 68)
print("DATA005 전송 요약")
print("=" * 68)
print(json.dumps(summary, ensure_ascii=False, indent=2))
if int(summary.get("remaining_bytes_after_run", 0)):
    print("\n다음 45 GB 구간이 남아 있습니다. 같은 노트북을 재실행하면 이어받습니다.")
else:
    print("\n✅ 선택한 공개 데이터셋의 Drive 보관이 완료되었습니다.")
